# Fase 0 — Setup do ambiente

Preparação do ambiente de execução do estudo comparativo **U-Net (CNN) vs SegFormer (ViT)** para segmentação semântica de cafezais.

Este estágio detecta a plataforma de execução, instala as dependências necessárias, carrega a configuração única do projeto, fixa as sementes, autentica o Google Earth Engine e registra o ambiente do run. As saídas são: ambiente pronto, configuração carregada e credenciais do GEE validadas.

## Detecção da raiz do repositório

Localiza a raiz do repositório a partir do diretório corrente e a insere no caminho de importação, garantindo o acesso ao pacote `src/`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


# Sobe os diretórios até encontrar src/config.yaml, marcador da raiz do projeto.
def _find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src" / "config.yaml").is_file():
            return candidate
    raise RuntimeError("Raiz do repositório não localizada (src/config.yaml ausente).")


PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Raiz do projeto: {PROJECT_ROOT}")

## Detecção da plataforma

Identifica o ambiente de execução (Kaggle, Colab ou local) para adaptar a instalação de dependências e a leitura de segredos.

In [ ]:
import importlib.util
import os


# Heurística por variáveis de ambiente e presença de diretórios característicos.
def detect_platform() -> str:
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or Path("/kaggle").is_dir():
        return "kaggle"
    if "COLAB_GPU" in os.environ or importlib.util.find_spec("google.colab") is not None:
        return "colab"
    return "local"


PLATFORM = detect_platform()
print(f"Plataforma detectada: {PLATFORM}")

## Instalação condicional das dependências

Em Kaggle/Colab instala o pacote com os extras geoespaciais e de aprendizado de máquina. No ambiente local a instalação é ignorada, pois é gerenciada por `uv` e pelo CI.

In [ ]:
import subprocess

# Instala o projeto editavelmente com os extras necessários apenas em nuvem.
if PLATFORM in {"kaggle", "colab"}:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[geo,ml]"],
        cwd=PROJECT_ROOT,
        check=True,
    )
    print("Dependências instaladas.")
else:
    print("Ambiente local: instalação ignorada (gerenciada por uv/CI).")

## Carregamento da configuração única

Lê a configuração de `src/config.yaml` por meio de `src/config.py`, fonte única de verdade de caminhos, bandas, parâmetros e sementes.

In [ ]:
from src.config import CONFIG

# Exibe um resumo dos parâmetros centrais do estudo.
print(f"Projeto: {CONFIG.get('project.name')} v{CONFIG.get('project.version')}")
print(f"Semente: {CONFIG.seed} | Patch: {CONFIG.patch_size} | Folds: {CONFIG.fold_count}")
print(f"Bandas: {CONFIG.bands}")
print(f"Hash da configuração: {CONFIG.config_hash}")

## Criação da árvore de diretórios

Garante a existência dos diretórios de dados, modelos e artefatos resolvidos a partir da configuração.

In [ ]:
# Cria os diretórios de trabalho, se ainda não existirem.
CONFIG.paths.ensure()
for name in ("data", "raw", "interim", "processed", "external", "models", "artifacts"):
    print(f"{name}: {getattr(CONFIG.paths, name)}")

## Fixação das sementes

Fixa as sementes de `python`, `numpy`, `torch` e `cuda` para garantir a reprodutibilidade dos experimentos.

In [ ]:
from src.config import seed_everything

# Aplica a semente global definida na configuração.
resolved_seed = seed_everything()
print(f"Sementes fixadas em {resolved_seed}.")

## Carregamento de segredos

Em Kaggle/Colab injeta os segredos do cofre da plataforma nas variáveis de ambiente esperadas pelo pacote. Nenhum valor é impresso. No ambiente local, os segredos devem vir de variáveis de ambiente ou do arquivo `.env`.

In [ ]:
# Nomes das variáveis de ambiente consumidas pelas fases seguintes.
SECRET_NAMES = (
    "GEE_SERVICE_ACCOUNT_EMAIL",
    "GEE_PROJECT",
    "GEE_SERVICE_ACCOUNT_KEY_JSON",
    "GEE_OAUTH_CREDENTIALS_JSON",
    "HF_TOKEN",
    "HF_USERNAME",
)

if PLATFORM == "kaggle":
    from kaggle_secrets import UserSecretsClient

    client = UserSecretsClient()
    for name in SECRET_NAMES:
        try:
            os.environ[name] = client.get_secret(name)
        except Exception:
            print(f"Segredo ausente no Kaggle: {name}")
elif PLATFORM == "colab":
    from google.colab import userdata

    for name in SECRET_NAMES:
        try:
            os.environ[name] = userdata.get(name)
        except Exception:
            print(f"Segredo ausente no Colab: {name}")
else:
    print("Ambiente local: segredos esperados via variáveis de ambiente/.env.")

## Autenticação no Google Earth Engine

Inicializa o Earth Engine com as credenciais lidas exclusivamente do ambiente: conta de serviço ou credenciais OAuth de usuário. A ausência de credenciais é reportada sem interromper a execução do notebook.

In [ ]:
from src.data.gee_client import GEECredentialsError, init_ee

# Inicializa o cliente do Earth Engine a partir das variáveis de ambiente.
try:
    ee = init_ee()
    GEE_READY = True
    print("Earth Engine autenticado com sucesso.")
except GEECredentialsError as exc:
    GEE_READY = False
    print(f"Credenciais do GEE ausentes: {exc}")

## Registro do ambiente do run

Persiste versões de dependências, plataforma, commit, hash da configuração e semente em `artifacts/environment.json`, sem expor credenciais.

In [ ]:
from src.config import log_environment, save_environment_log

# Salva o retrato do ambiente e exibe o conteúdo registrado.
log_path = save_environment_log()
print(f"Registro do ambiente salvo em: {log_path}")
log_environment()